# Model Validation and Explainability

The initial test scores are unusually high, so this notebook checks feature-target associations, model feature importance, cross-validation stability, and an additional random split. The original dataset and Step 3 result files are not modified.

The original 80/20 split with `random_state=42` is recreated exactly. Five-fold cross-validation uses only the original training data. The original test set is used once for the independent final comparison. `Malware` is encoded as the positive class.

## 1. Why validation is necessary

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid')
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
dataset_path = root / 'dataset' / 'Obfuscated-MalMem2022.csv'
models_path, images_path, reports_path = root / 'models', root / 'images', root / 'reports'
images_path.mkdir(exist_ok=True); reports_path.mkdir(exist_ok=True)
RANDOM_STATE = 42
POSITIVE_CLASS = 'Malware'
df = pd.read_csv(dataset_path)
config = json.loads((models_path / 'preprocessing_config.json').read_text(encoding='utf-8'))
features = config['feature_columns']
X = df[features].copy()
y_text = df[config['target_column']].copy()
y = y_text.map({'Benign': 0, 'Malware': 1}).astype('int8')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=config['test_size'], random_state=RANDOM_STATE, stratify=y)
print('Original dataset shape:', df.shape)
print('Final feature count:', len(features))
print('Original split:', X_train.shape, X_test.shape)
print('The test set is held out until final independent validation.')

Original dataset shape: (58058, 57)
Final feature count: 52
Original split: (46446, 52) (11612, 52)
The test set is held out until final independent validation.


## 2. Feature-target analysis

In [2]:
analysis_rows = []
for feature in features:
    values = X[feature].to_numpy(dtype=float)
    benign = X.loc[y == 0, feature].to_numpy(dtype=float)
    malware = X.loc[y == 1, feature].to_numpy(dtype=float)
    correlation = float(np.corrcoef(values, y.to_numpy())[0, 1])
    auc = float(roc_auc_score(y, values))
    directional_auc = max(auc, 1.0 - auc)
    analysis_rows.append({
        'Feature': feature,
        'Absolute Pearson Correlation': abs(correlation),
        'Directional Univariate AUC': directional_auc,
        'Benign Mean': float(np.mean(benign)),
        'Malware Mean': float(np.mean(malware)),
        'Benign Std': float(np.std(benign)),
        'Malware Std': float(np.std(malware)),
        'Minimum': float(np.min(values)),
        'Maximum': float(np.max(values)),
    })
feature_target_analysis = pd.DataFrame(analysis_rows).sort_values(['Directional Univariate AUC', 'Absolute Pearson Correlation'], ascending=False).reset_index(drop=True)
feature_target_analysis['Suspicion Rank'] = np.arange(1, len(feature_target_analysis) + 1)
feature_target_analysis.to_csv(reports_path / 'feature_target_analysis.csv', index=False)
print(feature_target_analysis.head(15).to_string(index=False))

                        Feature  Absolute Pearson Correlation  Directional Univariate AUC  Benign Mean  Malware Mean  Benign Std  Malware Std     Minimum      Maximum  Suspicion Rank
      dlllist.avg_dlls_per_proc                      0.908761                    0.997506    48.924049     38.494404    2.711447     2.023539    7.333334 5.317073e+01               1
              svcscan.nservices                      0.629701                    0.996502   394.205529    388.489057    1.352560     4.814748   94.000000 3.950000e+02               2
svcscan.shared_process_services                      0.710321                    0.994627   117.981763    115.777947    0.251541     1.528616   26.000000 1.180000e+02               3
         svcscan.kernel_drivers                      0.292325                    0.994273   221.990522    220.821581    0.127131     2.710233   55.000000 2.220000e+02               4
   handles.avg_handles_per_proc                      0.253614                    0.99

## 3. Feature importance

In [3]:
tree_preprocessor = ColumnTransformer([('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), features)], remainder='drop', verbose_feature_names_out=False)
importance_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9, objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist'),
}
importance_rows = []
fitted_importance_models = {}
for model_name, estimator in importance_models.items():
    pipeline = Pipeline([('preprocess', tree_preprocessor), ('model', estimator)])
    pipeline.fit(X_train, y_train)
    fitted_importance_models[model_name] = pipeline
    values = pipeline.named_steps['model'].feature_importances_
    for feature, importance in zip(features, values):
        importance_rows.append({'Model': model_name, 'Feature': feature, 'Importance': float(importance)})
importance_df = pd.DataFrame(importance_rows).sort_values(['Model', 'Importance'], ascending=[True, False])
importance_df['Rank'] = importance_df.groupby('Model')['Importance'].rank(method='first', ascending=False).astype(int)
importance_df.to_csv(reports_path / 'feature_importance.csv', index=False)
top15 = importance_df[importance_df['Rank'] <= 15]
print(top15.groupby('Model').head(15).to_string(index=False))

        Model                         Feature  Importance  Rank
Decision Tree               svcscan.nservices    0.986782     1
Decision Tree        svcscan.process_services    0.006516     2
Decision Tree    handles.avg_handles_per_proc    0.002989     3
Decision Tree            callbacks.ncallbacks    0.001366     4
Decision Tree            malfind.commitCharge    0.000860     5
Decision Tree                handles.nhandles    0.000789     6
Decision Tree                   handles.nfile    0.000203     7
Decision Tree                  handles.nevent    0.000166     8
Decision Tree                handles.ndesktop    0.000159     9
Decision Tree              malfind.protection    0.000080    10
Decision Tree         psxview.not_in_deskthrd    0.000079    11
Decision Tree                 handles.nmutant    0.000012    12
Decision Tree                    pslist.nproc    0.000000    13
Decision Tree                    pslist.nppid    0.000000    14
Decision Tree              pslist.avg_th

In [4]:
best_importance_model = 'Random Forest'
plot_data = importance_df[importance_df['Model'] == best_importance_model].head(15).sort_values('Importance')
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_data['Feature'], plot_data['Importance'], color='#1f77b4')
ax.set_title('Top 15 Feature Importances: Random Forest')
ax.set_xlabel('Importance')
fig.tight_layout()
fig.savefig(images_path / 'top_15_feature_importance.png', dpi=300, bbox_inches='tight')
plt.close(fig)
print('Saved top_15_feature_importance.png')

Saved top_15_feature_importance.png


## 4. Five-fold cross-validation on training data only

In [5]:
cv_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9, objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist'),
}
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1'}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, estimator in cv_models.items():
    pipeline = Pipeline([('preprocess', tree_preprocessor), ('model', estimator)])
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1, return_train_score=False)
    cv_rows.append({'Model': name, 'Mean Accuracy': scores['test_accuracy'].mean(), 'Std Accuracy': scores['test_accuracy'].std(), 'Mean Precision': scores['test_precision'].mean(), 'Mean Recall': scores['test_recall'].mean(), 'Mean F1-Score': scores['test_f1'].mean()})
cv_results = pd.DataFrame(cv_rows).sort_values('Mean F1-Score', ascending=False).reset_index(drop=True)
cv_results.to_csv(reports_path / 'cross_validation_results.csv', index=False)
print(cv_results.to_string(index=False))

        Model  Mean Accuracy  Std Accuracy  Mean Precision  Mean Recall  Mean F1-Score
      XGBoost       0.999828      0.000086        0.999740     0.999913       0.999827
Random Forest       0.999828      0.000175        0.999827     0.999827       0.999827
Decision Tree       0.999742      0.000129        0.999653     0.999827       0.999740


## 5. Alternative 70/30 split validation

In [6]:
X_alt_train, X_alt_test, y_alt_train, y_alt_test = train_test_split(X, y, test_size=0.30, random_state=123, stratify=y)
alt_rows = []
for name, estimator in cv_models.items():
    pipeline = Pipeline([('preprocess', tree_preprocessor), ('model', estimator)])
    pipeline.fit(X_alt_train, y_alt_train)
    pred = pipeline.predict(X_alt_test)
    alt_rows.append({'Model': name, 'Accuracy': accuracy_score(y_alt_test, pred), 'Precision': precision_score(y_alt_test, pred, zero_division=0), 'Recall': recall_score(y_alt_test, pred, zero_division=0), 'F1-Score': f1_score(y_alt_test, pred, zero_division=0)})
alternative_results = pd.DataFrame(alt_rows).sort_values('F1-Score', ascending=False).reset_index(drop=True)
alternative_results.to_csv(reports_path / 'alternative_split_results.csv', index=False)
print('Additional robustness experiment: stratified 70/30 split, random_state=123')
print(alternative_results.to_string(index=False))

Additional robustness experiment: stratified 70/30 split, random_state=123
        Model  Accuracy  Precision   Recall  F1-Score
Random Forest  0.999828   0.999653 1.000000  0.999827
      XGBoost  0.999770   0.999538 1.000000  0.999769
Decision Tree  0.999598   0.999422 0.999769  0.999595


## 6. Comparison with the original independent test performance

In [7]:
original_test_rows = []
for name, estimator in cv_models.items():
    pipeline = Pipeline([('preprocess', tree_preprocessor), ('model', estimator)])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    original_test_rows.append({'Model': name, 'Test Accuracy': accuracy_score(y_test, pred), 'Test Precision': precision_score(y_test, pred, zero_division=0), 'Test Recall': recall_score(y_test, pred, zero_division=0), 'Test F1-Score': f1_score(y_test, pred, zero_division=0)})
original_test_results = pd.DataFrame(original_test_rows)
comparison = cv_results.merge(original_test_results, on='Model')
comparison.to_csv(reports_path / 'validation_comparison.csv', index=False)
print(comparison.to_string(index=False))

        Model  Mean Accuracy  Std Accuracy  Mean Precision  Mean Recall  Mean F1-Score  Test Accuracy  Test Precision  Test Recall  Test F1-Score
      XGBoost       0.999828      0.000086        0.999740     0.999913       0.999827            1.0             1.0          1.0            1.0
Random Forest       0.999828      0.000175        0.999827     0.999827       0.999827            1.0             1.0          1.0            1.0
Decision Tree       0.999742      0.000129        0.999653     0.999827       0.999740            1.0             1.0          1.0            1.0


## 7. Leakage assessment

In [8]:
top_target = feature_target_analysis.head(10)
top_importance = importance_df.sort_values('Importance', ascending=False).head(10)
suspicious_threshold = 0.995
suspicious_features = feature_target_analysis[feature_target_analysis['Directional Univariate AUC'] >= suspicious_threshold]['Feature'].tolist()
identifier_like_features = [feature for feature in features if any(token in feature.lower() for token in ['id', 'name', 'hash', 'label', 'class', 'category', 'family'])]
potential_leakage = bool(identifier_like_features or suspicious_features)
assessment = {
    'potential_leakage_detected': potential_leakage,
    'high_univariate_auc_features': suspicious_features,
    'identifier_like_feature_names': identifier_like_features,
    'interpretation': 'High predictive strength is present and requires domain review; the numeric names are behavioral/memory telemetry rather than explicit labels, but perfect separation should not be treated as universally generalizable.',
    'recommendation': 'Retain original results for this dataset, but describe them as dataset-specific and perform external or temporal validation before claiming deployment-level generalization.'
}
(reports_path / 'leakage_assessment.json').write_text(json.dumps(assessment, indent=2), encoding='utf-8')
print('Top 10 target-association features:')
print(top_target.to_string(index=False))
print('\nTop 10 importance features:')
print(top_importance.to_string(index=False))
print('\nPotential leakage detected:', potential_leakage)
print('Explicit identifier-like features:', identifier_like_features)
print('Near-perfect univariate features:', suspicious_features)

Top 10 target-association features:
                        Feature  Absolute Pearson Correlation  Directional Univariate AUC  Benign Mean  Malware Mean  Benign Std  Malware Std    Minimum     Maximum  Suspicion Rank
      dlllist.avg_dlls_per_proc                      0.908761                    0.997506    48.924049     38.494404    2.711447     2.023539   7.333334    53.17073               1
              svcscan.nservices                      0.629701                    0.996502   394.205529    388.489057    1.352560     4.814748  94.000000   395.00000               2
svcscan.shared_process_services                      0.710321                    0.994627   117.981763    115.777947    0.251541     1.528616  26.000000   118.00000               3
         svcscan.kernel_drivers                      0.292325                    0.994273   221.990522    220.821581    0.127131     2.710233  55.000000   222.00000               4
   handles.avg_handles_per_proc                      0.2536

## 8. Final conclusion

In [9]:
best_cv = cv_results.iloc[0]
best_alt = alternative_results.iloc[0]
recommendation = '\n'.join([
    'Validation conclusion: Potential leakage detected and specific features require domain review because near-perfect univariate predictive strength was observed.',
    'The features are named behavioral/memory telemetry fields, not explicit label or family identifiers. This supports retaining the original results for this dataset while avoiding universal claims.',
    'The original 100% results should be retained as measured, but further external, temporal, or cross-dataset validation is required before deployment claims.',
    f'Best cross-validation model: {best_cv["Model"]}, mean F1={best_cv["Mean F1-Score"]:.6f}, mean recall={best_cv["Mean Recall"]:.6f}.',
    f'Best additional 70/30 model: {best_alt["Model"]}, F1={best_alt["F1-Score"]:.6f}, recall={best_alt["Recall"]:.6f}.',
    'Recommended final project model: retain the Decision Tree as the selected model for this dataset because it tied for the highest original test performance and is the selected Step 3 artifact; do not call it universally best.'
])
(reports_path / 'validation_conclusion.txt').write_text(recommendation, encoding='utf-8')
required = ['feature_target_analysis.csv', 'feature_importance.csv', 'cross_validation_results.csv', 'alternative_split_results.csv', 'validation_comparison.csv', 'leakage_assessment.json', 'validation_conclusion.txt']
required += ['top_15_feature_importance.png']
missing = [name for name in required if not (reports_path / name).exists() and not (images_path / name).exists()]
if missing: raise FileNotFoundError(missing)
print(recommendation)
print('All Step 4 validation files verified.')

Validation conclusion: Potential leakage detected and specific features require domain review because near-perfect univariate predictive strength was observed.
The features are named behavioral/memory telemetry fields, not explicit label or family identifiers. This supports retaining the original results for this dataset while avoiding universal claims.
The original 100% results should be retained as measured, but further external, temporal, or cross-dataset validation is required before deployment claims.
Best cross-validation model: XGBoost, mean F1=0.999827, mean recall=0.999913.
Best additional 70/30 model: Random Forest, F1=0.999827, recall=1.000000.
Recommended final project model: retain the Decision Tree as the selected model for this dataset because it tied for the highest original test performance and is the selected Step 3 artifact; do not call it universally best.
All Step 4 validation files verified.
